# Kuramoto Benchmark

Goal: evaluate continuous-state MF-REINFORCE on synchronization control for interacting oscillators.

**Environment Basics**

- State space: phases on the circle, represented by lifted phases $\theta_i\in\mathbb{R}$ during simulation and wrapped phases $\theta_i\bmod 2\pi$ for diagnostics.
- Action space: $\mathcal{A}=\mathbb{R}$, with scalar angular control $u_i$; the policy mean is bounded by `a_max` and rollout exploration adds Gaussian noise.
- Population law signature: first Fourier moment $(C_t,S_t)$, equivalently the order parameter $R_t$.
- Dynamics: $\theta_{i,t+1}=\theta_{i,t}+\Delta t(\omega_i+K\,A_i+u_i)+\sqrt{2D\Delta t}\xi_{i,t+1}$, where $A_i$ is the mean-field sine interaction.
- Main task: synchronize phases and align the population with the target phase $\theta_\star$.

The central statistic is the order parameter

$$
R_t=\left|\frac{1}{N}\sum_{j=1}^N e^{i\theta_j(t)}\right|,
$$

with target-aligned order

$$
A_t=\cos(\theta_\star)C_t+\sin(\theta_\star)S_t,
\quad
C_t=\frac{1}{N}\sum_j\cos\theta_j(t),\quad
S_t=\frac{1}{N}\sum_j\sin\theta_j(t).
$$

The figures track synchronization, target locking, phase snapshots, post-control persistence, and control energy.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from mfc.experiments import notebook_helpers as nh

ENV_NAME = "kuramoto"
BASE_DIR = ROOT / "runs" / "notebook_bundles" / ENV_NAME
PRESET = "smoke"
QUICK = PRESET == "smoke"
RUN_MISSING = True
FORCE_REBUILD = False
EXTENDED = True

In [ ]:
bundle = (
    nh.ensure_continuous_benchmark_bundle(ENV_NAME, BASE_DIR, quick=QUICK, force=FORCE_REBUILD, extended=EXTENDED, preset=PRESET)
    if RUN_MISSING
    else nh.continuous_bundle_paths(ENV_NAME, BASE_DIR)
)
bundle

## Figure Coverage

Goal: show which requested results from `docs/figures.md` are currently produced by this notebook and which artifacts support them.

The tables are an audit layer, not an estimator. A row maps a desired result family to the command or study that generates its data and to the notebook helper that renders it.

In [ ]:
nh.figure_checklist(ENV_NAME)

In [ ]:
nh.figure_coverage_matrix(ENV_NAME)

## Training

Goal: summarize the continuous-state MF-REINFORCE optimization trace.

The value/cost curve reports the evaluation objective at episode $k$, and the gradient-norm curve reports

$$
\|\widehat g_k\|_2,
$$

where $\widehat g_k$ is the continuous MF-REINFORCE estimator used for that parameter update.

In [ ]:
histories = nh.load_training_histories(bundle)
nh.plot_training_comparison(histories)

## Application Diagnostics

Goal: determine whether the learned controller synchronizes phases and locks them to the target direction.

Reference: application plots compare the learned controller with uncontrolled dynamics and, when configured, a grid-searched synchronization/target-locking heuristic. Gradient/sensitivity diagnostic sections use independent pathwise-AD particle references.

The key diagnostics are the order parameter $R_t$, target-aligned order $A_t$, circular synchronization cost, and mean control energy $\frac{1}{N}\sum_i |u_i(t)|^2$. Circle snapshots show phase concentration at selected times, and post-control curves test persistence.

In [ ]:
application = nh.load_application_data(bundle)
display(nh.reference_solution_table(ENV_NAME, application))
nh.plot_continuous_time_metrics(application, ENV_NAME)
nh.plot_continuous_policy_and_samples(application, ENV_NAME)
nh.plot_continuous_snapshots(application, ENV_NAME)
nh.plot_continuous_application_details(application, ENV_NAME)

## Universal Diagnostics

Goal: validate the estimator chain before interpreting optimization performance.

The perturbation plots measure empirical geometry,

$$
d(M^\lambda,\mu),
$$

including quantile bands and local log-log slopes. The functional-law plots study

$$
\Gamma(M^\lambda)=(F_1(M^\lambda),\ldots,F_k(M^\lambda)),
\qquad
\frac{\Gamma(M^\lambda)-\Gamma(\mu)}{\lambda},
$$

which is the induced law of population signatures. The score plots check

$$
S_{t,\lambda}^\theta=\nabla_\theta\log q_{t,\lambda}^\theta(M_t),
\qquad \mathbb{E}[S_{t,\lambda}^\theta]\approx 0,
$$

and the gradient plots report bias, variance, MSE, norm ratio, and cosine agreement for an estimator $\widehat g$ against an oracle or reference gradient $g$:

$$
\operatorname{MSE}=\mathbb{E}\|\widehat g-g\|_2^2,
\qquad
\cos(\widehat g,g)=\frac{\widehat g\cdot g}{\|\widehat g\|_2\|g\|_2}.
$$

The sensitivity plots track errors in $D_t=\partial_\theta\Gamma(\mu_t^\theta)$, or in the finite-state case $D_t=\partial_\theta\mu_t^\theta$.

In [ ]:
diagnostics = nh.load_diagnostic_data(bundle)
nh.plot_perturbation_geometry(diagnostics)
nh.plot_perturbation_slopes(diagnostics)
nh.plot_functional_law(diagnostics)
nh.plot_functional_signature_means(diagnostics)
nh.plot_score_validation(diagnostics)
nh.plot_score_coordinate_diagnostics(diagnostics)
nh.plot_gradient_validation(diagnostics)
nh.plot_gradient_error_decomposition(diagnostics)
nh.plot_sensitivity_validation(diagnostics)
nh.plot_sensitivity_heatmap(diagnostics)

## Scaling, Budget, And Optimization Summaries

Goal: measure how estimator quality and optimization performance change with simulator budget, auxiliary budget, horizon, and training time.

The budget heatmap varies main and auxiliary samples $(B,n)$ under the approximate cost model

$$
C\approx C_{main}B+C_{aux}n.
$$

The horizon plot studies how gradient error changes with $T$, and the optimization plots compare objective or cost gaps against iteration count, runtime, and simulator-call budget.

In [ ]:
studies = nh.load_study_data(bundle)
grid_metrics = nh.load_study_grid_metrics(bundle)
optimization_history = nh.load_optimization_history(bundle)
nh.plot_budget_and_horizon(studies)
nh.plot_budget_pareto(studies, grid_metrics)
nh.plot_optimization_history(optimization_history)
nh.plot_optimization_summary(studies)

## Raw Tables For Custom Figures

Goal: expose the underlying CSV/JSON artifacts used by the plots so paper figures can be restyled or recomputed without rerunning training.

These tables are not new estimators. They are the saved values for population flows, policies, diagnostics, study grids, histories, and final metrics.

In [ ]:
application[nh.CONTINUOUS_ALGORITHM]["time_metrics"].head(), diagnostics[nh.CONTINUOUS_ALGORITHM]["gradient"].head(), studies["budget"].head()